# Notebook 1: CUDA Foundations on A100

This notebook starts with one thread and adds one idea at a time. Do not use Run All on your first attempt. Read, predict, and then run each cell.


In [ ]:
from pathlib import Path
import shutil
import subprocess

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "CMakeLists.txt").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the cuda-kernels-a100-beginners repository")

ROOT = find_repo_root(Path.cwd())
required = ("nvidia-smi", "nvcc", "cmake")
missing = [command for command in required if shutil.which(command) is None]
assert not missing, f"Missing required commands: {missing}"
print(subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True, check=True).stdout)


## Build

The build targets `sm_80`. This cell builds only the foundations track.


In [ ]:
subprocess.run([
    "cmake", "-S", str(ROOT), "-B", str(ROOT / "build"),
    "-DCMAKE_BUILD_TYPE=Release", "-DCMAKE_CUDA_ARCHITECTURES=80"
], check=True)
targets = [
    "00_device_query", "01_hello_kernel", "02_one_block_index",
    "03_global_index", "04_bounds_check", "05_memory_roundtrip", "06_vector_add",
]
subprocess.run(["cmake", "--build", str(ROOT / "build"), "--target", *targets, "-j"], check=True)


## Step 1: The CPU Launches a Kernel on the GPU

Before running: how many times will `Hello from the GPU` be printed?


In [ ]:
result = subprocess.run([str(ROOT / "build/01_hello_kernel")], text=True, capture_output=True)
print(result.stdout, result.stderr)
assert result.returncode == 0 and "PASS hello_kernel" in result.stdout


## Step 2: Four Threads Inside One Block

Predict the four `threadIdx.x` values. The print order may vary.


In [ ]:
result = subprocess.run([str(ROOT / "build/02_one_block_index")], text=True, capture_output=True)
print(result.stdout, result.stderr)
assert result.returncode == 0 and "PASS one_block_index" in result.stdout


## Step 3: Two Blocks and a Global Index

The formula is `blockIdx.x * blockDim.x + threadIdx.x`. Compute the index of thread 2 in block 1 before running.


In [ ]:
result = subprocess.run([str(ROOT / "build/03_global_index")], text=True, capture_output=True)
print(result.stdout, result.stderr)
assert result.returncode == 0 and "PASS global_index" in result.stdout


## Step 4: Six Values, Eight Threads

Two threads are extra. The guard prevents them from touching memory outside the array.


In [ ]:
result = subprocess.run([str(ROOT / "build/04_bounds_check")], text=True, capture_output=True)
print(result.stdout, result.stderr)
assert result.returncode == 0 and "PASS bounds_check" in result.stdout


## Step 5: CPU and GPU Memory

The vector starts on the CPU, is copied to the GPU, changed by a kernel, and copied back to the CPU.


In [ ]:
result = subprocess.run([str(ROOT / "build/05_memory_roundtrip")], text=True, capture_output=True)
print(result.stdout, result.stderr)
assert result.returncode == 0 and "PASS memory_roundtrip" in result.stdout


## Step 6: Vector Addition and a Residual Connection

Each thread adds one element. In an LLM, this is the same pattern as `hidden = layer_output + residual`.


In [ ]:
result = subprocess.run([str(ROOT / "build/06_vector_add")], text=True, capture_output=True)
print(result.stdout, result.stderr)
assert result.returncode == 0 and "PASS vector_add" in result.stdout


## Final Check

Explain aloud:

1. What is the difference between a block and a thread?
2. Why does indexing start at 0?
3. Why are there extra threads?
4. Where is the data before and after `cudaMemcpy`?
5. How is vector addition related to a residual connection?
